# Satellite Data Cleaning and Curation

This notebook prepares the satellite dataset for downstream machine-learning analysis.

The exploratory analysis identified several data-quality issues, including duplicate records in the auxiliary UCS dataset, incomplete orbital information, heterogeneous categorical variables, and potentially unreliable cross-dataset matches.

The goals of this notebook are to:

1. define the satellite population used for modeling;
2. validate the integration between the orbital catalog and the UCS satellite database;
3. select and clean relevant modeling variables;
4. characterize and handle missing data without introducing target leakage;
5. create reproducible training, validation, and test datasets.

Preprocessing steps that learn parameters from the data, such as imputation, scaling, and categorical encoding, are fitted using the training data only.

## Imports and Configuration

In [1]:
from pathlib import Path

import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder, OrdinalEncoder

pd.options.display.max_columns = None

RANDOM_SEED = 42

## Data Loading

We use the same versioned source data as in the exploratory analysis so that all notebooks operate on a consistent snapshot of the catalog.

The orbital catalog contains several object types, while the auxiliary UCS database provides satellite-specific metadata. Since the downstream prediction task concerns reported satellite lifetime, this notebook focuses on `PAYLOAD` objects before integrating the UCS data.

In [2]:
SOURCE_COMMIT = "1c49c73b92e84d3eba2c53f43313ed861d08f896"

BASE_URL = (
    "https://raw.githubusercontent.com/"
    f"EnzoRg/space_debris/{SOURCE_COMMIT}/data/raw"
)

satellites_df = pd.read_json(f"{BASE_URL}/satellites.json")
extra_data_df = pd.read_excel(
    f"{BASE_URL}/ucs-satellite-database.xlsx"
)

print(f"Satellite catalog: {satellites_df.shape}")
print(f"UCS database:      {extra_data_df.shape}")

Satellite catalog: (20328, 24)
UCS database:      (7560, 68)


## Modeling Population

The prediction task is restricted to catalog entries classified as `PAYLOAD`.

This avoids combining fundamentally different object types and makes the population consistent with the auxiliary UCS database, which contains satellite-specific attributes such as purpose, launch mass, and expected lifetime.

In [3]:
print(satellites_df["OBJECT_TYPE"].value_counts())

assert satellites_df["OBJECT_TYPE"].eq("PAYLOAD").all()
assert satellites_df["NORAD_CAT_ID"].is_unique

print(f"Satellite records: {len(satellites_df):,}")
print(f"Unique NORAD IDs:  {satellites_df['NORAD_CAT_ID'].nunique():,}")

OBJECT_TYPE
PAYLOAD    20328
Name: count, dtype: int64
Satellite records: 20,328
Unique NORAD IDs:  20,328


## Cleaning the UCS Database

Before merging the two sources, we ensure that the UCS database contains a unique record for each NORAD identifier.

Exact duplicate rows can be removed directly. If the same NORAD identifier is associated with multiple non-identical UCS records, the ambiguity is resolved by comparing the reported satellite name with the corresponding name in the orbital catalog.

In [4]:
UCS_ID = "NORAD Number"

ucs_duplicate_summary = pd.Series({
    "Rows": len(extra_data_df),
    "Missing NORAD IDs": extra_data_df[UCS_ID].isna().sum(),
    "Unique NORAD IDs": extra_data_df[UCS_ID].nunique(),
    "Rows belonging to repeated NORAD IDs": (
        extra_data_df[UCS_ID].duplicated(keep=False).sum()
    ),
    "Exact duplicate rows beyond first occurrence": (
        extra_data_df.duplicated().sum()
    ),
})

display(ucs_duplicate_summary.to_frame("Value"))

,Value
Rows,7560
Missing NORAD IDs,0
Unique NORAD IDs,7551
Rows belonging to repeated NORAD IDs,18
Exact duplicate rows beyond first occurrence,7


In [5]:
ucs_clean_df = extra_data_df.drop_duplicates().copy()

repeated_id_mask = ucs_clean_df[UCS_ID].duplicated(keep=False)

ucs_conflicts = (
    ucs_clean_df.loc[
        repeated_id_mask,
        [
            UCS_ID,
            "Name of Satellite, Alternate Names",
            "Country of Operator/Owner",
            "Purpose",
        ],
    ]
    .sort_values(UCS_ID)
)

display(ucs_conflicts)

,NORAD Number,"Name of Satellite, Alternate Names",Country of Operator/Owner,Purpose
6621,54766,Starlink-5471,USA,Communications
6622,54766,Starlink-5472,USA,Communications
6744,55455,Starlink-5636,USA,Communications
6787,55455,Starlink-5680,USA,Communications


In [6]:
catalog_names = satellites_df.set_index("NORAD_CAT_ID")["SATNAME"]

duplicate_candidates = ucs_clean_df.loc[repeated_id_mask].copy()
duplicate_candidates["CATALOG_NAME"] = duplicate_candidates[UCS_ID].map(
    catalog_names
)

duplicate_candidates["NAME_MATCH"] = (
    duplicate_candidates["Name of Satellite, Alternate Names"]
    .astype("string")
    .str.strip()
    .str.casefold()
    .eq(
        duplicate_candidates["CATALOG_NAME"]
        .astype("string")
        .str.strip()
        .str.casefold()
    )
    .fillna(False)
)

display(
    duplicate_candidates[
        [
            UCS_ID,
            "Name of Satellite, Alternate Names",
            "CATALOG_NAME",
            "NAME_MATCH",
        ]
    ]
)

,NORAD Number,"Name of Satellite, Alternate Names",CATALOG_NAME,NAME_MATCH
6621,54766,Starlink-5471,STARLINK-5471,True
6622,54766,Starlink-5472,STARLINK-5471,False
6744,55455,Starlink-5636,STARLINK-5680,False
6787,55455,Starlink-5680,STARLINK-5680,True


In [7]:
matches_per_duplicate_id = (
    duplicate_candidates
    .groupby(UCS_ID)["NAME_MATCH"]
    .sum()
)

display(
    matches_per_duplicate_id
    .rename("Number of matching UCS records")
    .to_frame()
)

assert matches_per_duplicate_id.eq(1).all()

,Number of matching UCS records
NORAD Number,
54766,1
55455,1


In [8]:
non_conflicting_ucs = ucs_clean_df.loc[~repeated_id_mask]

resolved_conflicts = duplicate_candidates.loc[
    duplicate_candidates["NAME_MATCH"]
].drop(columns=["CATALOG_NAME", "NAME_MATCH"])

ucs_unique_df = pd.concat(
    [non_conflicting_ucs, resolved_conflicts],
    ignore_index=True,
)

assert ucs_unique_df[UCS_ID].is_unique

print(f"Clean UCS records: {len(ucs_unique_df):,}")
print(f"Unique NORAD IDs:  {ucs_unique_df[UCS_ID].nunique():,}")

Clean UCS records: 7,551
Unique NORAD IDs:  7,551


## Cross-Dataset Match Validation

The two datasets are linked through the NORAD catalog identifier.

Rather than assuming that every NORAD match is automatically reliable, we first measure the coverage of the UCS database within the payload catalog and inspect name consistency among matched records.

In [9]:
match_audit_df = satellites_df[
    ["NORAD_CAT_ID", "SATNAME"]
].merge(
    ucs_unique_df[
        [
            "NORAD Number",
            "Name of Satellite, Alternate Names",
        ]
    ],
    how="left",
    left_on="NORAD_CAT_ID",
    right_on="NORAD Number",
    validate="one_to_one",
    indicator=True,
)

matched_satellites = (match_audit_df["_merge"] == "both").sum()
match_rate = 100 * matched_satellites / len(match_audit_df)

print(f"Payload catalog records: {len(match_audit_df):,}")
print(f"Matched with UCS:         {matched_satellites:,}")
print(f"Match coverage:           {match_rate:.2f}%")

Payload catalog records: 20,328
Matched with UCS:         7,369
Match coverage:           36.25%


In [10]:
matched_df = match_audit_df.loc[
    match_audit_df["_merge"] == "both"
].copy()

catalog_name_normalized = (
    matched_df["SATNAME"]
    .astype("string")
    .str.strip()
    .str.casefold()
)

ucs_name_normalized = (
    matched_df["Name of Satellite, Alternate Names"]
    .astype("string")
    .str.strip()
    .str.casefold()
)

matched_df["EXACT_NAME_MATCH"] = (
    catalog_name_normalized == ucs_name_normalized
)

display(
    matched_df["EXACT_NAME_MATCH"]
    .value_counts()
    .rename("Records")
    .to_frame()
)

,Records
EXACT_NAME_MATCH,
True,5028
False,2341


In [11]:
name_mismatches = matched_df.loc[
    ~matched_df["EXACT_NAME_MATCH"],
    [
        "NORAD_CAT_ID",
        "SATNAME",
        "Name of Satellite, Alternate Names",
    ],
]

print(f"Name mismatches: {len(name_mismatches):,}")
display(name_mismatches.head(10))

Name mismatches: 2,341


,NORAD_CAT_ID,SATNAME,"Name of Satellite, Alternate Names"
1942,26473,USA 152,"Lacrosse/Onyx 4 (Lacrosse-4, 152)"
2010,31797,SAR LUPE 2,SAR-Lupe 2
2015,32750,SAR LUPE 4,SAR-Lupe 4
2016,33244,SAR LUPE 5,SAR-Lupe 5
2034,37813,IGS 6A,"IGS-6A (Information Gathering Satellite 6A, IG..."
2036,37954,IGS 7A,"IGS-7A (Information Gathering Satellite 7A, IG..."
2519,42072,IGS R-5,IGS Radar 5 (Information Gathering Satellite 10A)
2540,26694,SICRAL 1,Sicral 1A
2541,29658,SAR LUPE 1,SAR-Lupe 1
2543,39061,IGS 8A,"IGS-8A (Information Gathering Satellite 8A, IG..."


### Validating Identifier Matches

Exact satellite-name equality is too restrictive because the two sources frequently use different formatting or alternate names.

NORAD therefore remains the primary link between the datasets, while the orbital catalog's international designator (`INTLDES`) and the UCS `COSPAR Number` provide an independent identifier check. Launch dates are used as an additional diagnostic rather than as a strict matching criterion.

In [12]:
match_validation_df = satellites_df[
    [
        "NORAD_CAT_ID",
        "SATNAME",
        "INTLDES",
        "LAUNCH",
    ]
].merge(
    ucs_unique_df[
        [
            "NORAD Number",
            "Name of Satellite, Alternate Names",
            "COSPAR Number",
            "Date of Launch",
        ]
    ],
    how="inner",
    left_on="NORAD_CAT_ID",
    right_on="NORAD Number",
    validate="one_to_one",
)

print(f"Matched records: {len(match_validation_df):,}")

Matched records: 7,369


In [13]:
def normalize_identifier(series):
    return (
        series
        .astype("string")
        .str.strip()
        .str.upper()
        .str.replace(" ", "", regex=False)
    )


match_validation_df["INTLDES_NORM"] = normalize_identifier(
    match_validation_df["INTLDES"]
)

match_validation_df["COSPAR_NORM"] = normalize_identifier(
    match_validation_df["COSPAR Number"]
)

match_validation_df["COSPAR_MATCH"] = (
    match_validation_df["INTLDES_NORM"]
    .eq(match_validation_df["COSPAR_NORM"])
    .fillna(False)
)

In [14]:
cospar_summary = pd.Series({
    "Matched records": len(match_validation_df),
    "Catalog INTLDES available": match_validation_df["INTLDES"].notna().sum(),
    "UCS COSPAR available": match_validation_df["COSPAR Number"].notna().sum(),
    "COSPAR matches": match_validation_df["COSPAR_MATCH"].sum(),
    "COSPAR mismatches": (~match_validation_df["COSPAR_MATCH"]).sum(),
})

display(cospar_summary.to_frame("Records"))

,Records
Matched records,7369
Catalog INTLDES available,7369
UCS COSPAR available,7369
COSPAR matches,7270
COSPAR mismatches,99


In [15]:
match_validation_df["CATALOG_LAUNCH_DATE"] = pd.to_datetime(
    match_validation_df["LAUNCH"],
    errors="coerce",
).dt.normalize()

match_validation_df["UCS_LAUNCH_DATE"] = pd.to_datetime(
    match_validation_df["Date of Launch"],
    errors="coerce",
).dt.normalize()

match_validation_df["LAUNCH_DATE_MATCH"] = (
    match_validation_df["CATALOG_LAUNCH_DATE"]
    .eq(match_validation_df["UCS_LAUNCH_DATE"])
    .fillna(False)
)

validation_summary = pd.crosstab(
    match_validation_df["COSPAR_MATCH"],
    match_validation_df["LAUNCH_DATE_MATCH"],
)

validation_summary.index.name = "COSPAR match"
validation_summary.columns.name = "Launch-date match"

display(validation_summary)

Launch-date match,False,True
COSPAR match,,
False,4,95
True,623,6647


The cross-dataset validation shows that 7,270 of the 7,369 NORAD-based matches (98.66%) also agree on the international designator.

Launch dates are less consistent across the two sources: 623 records have matching NORAD and COSPAR identifiers but different recorded launch dates. Therefore, launch date is treated as a diagnostic variable rather than as a strict matching criterion.

For the curated dataset, we retain only records for which both the NORAD identifier and the international designator agree across sources. This conservative rule removes 99 ambiguous matches while preserving the vast majority of the linked satellite population.

In [16]:
validated_matches = match_validation_df.loc[
    match_validation_df["COSPAR_MATCH"]
].copy()

print(f"NORAD-based matches:       {len(match_validation_df):,}")
print(f"Validated matches:         {len(validated_matches):,}")
print(
    f"Retained after validation: "
    f"{100 * len(validated_matches) / len(match_validation_df):.2f}%"
)

NORAD-based matches:       7,369
Validated matches:         7,270
Retained after validation: 98.66%


## Curated Satellite Dataset

After validating the cross-dataset links, we construct the working dataset by combining orbital-catalog information with satellite-specific metadata from UCS.

Only NORAD matches independently confirmed by the international designator are retained.

In [17]:
validated_norad_ids = validated_matches["NORAD_CAT_ID"]

curated_satellites_df = (
    satellites_df[
        satellites_df["NORAD_CAT_ID"].isin(validated_norad_ids)
    ]
    .merge(
        ucs_unique_df,
        how="inner",
        left_on="NORAD_CAT_ID",
        right_on="NORAD Number",
        validate="one_to_one",
    )
)

assert len(curated_satellites_df) == len(validated_matches)
assert curated_satellites_df["NORAD_CAT_ID"].is_unique

print(f"Curated satellite records: {len(curated_satellites_df):,}")
print(f"Columns:                   {curated_satellites_df.shape[1]:,}")

Curated satellite records: 7,270
Columns:                   92


## Target Variable

The downstream supervised-learning task predicts the reported expected-lifetime category of a satellite.

`Expected Lifetime (yrs.)` is therefore treated as the target variable. Missing target values are not imputed: records without a reported expected lifetime cannot provide supervised-learning labels and are excluded from the labeled modeling population.

In [18]:
TARGET = "Expected Lifetime (yrs.)"

target_summary = pd.Series({
    "Curated records": len(curated_satellites_df),
    "Target available": curated_satellites_df[TARGET].notna().sum(),
    "Target missing": curated_satellites_df[TARGET].isna().sum(),
    "Unique target values": curated_satellites_df[TARGET].nunique(),
})

display(target_summary.to_frame("Value"))

display(
    curated_satellites_df[TARGET]
    .describe()
    .to_frame("Expected lifetime (years)")
)

,Value
Curated records,7270
Target available,5351
Target missing,1919
Unique target values,28


,Expected lifetime (years)
count,5351.00000
mean,5.36596
std,3.44106
min,0.25000
25%,4.00000
50%,4.00000
75%,5.00000
max,30.00000


In [19]:
display(
    curated_satellites_df[TARGET]
    .value_counts(dropna=False)
    .sort_index()
    .rename("Satellites")
    .to_frame()
)

,Satellites
Expected Lifetime (yrs.),
0.25,1
0.50,3
1.00,23
1.50,1
2.00,130
2.50,1
3.00,266
4.00,3348
4.50,1


The reported expected lifetime is available for 5,351 of the 7,270 validated satellite records.

The target is strongly discrete rather than continuously distributed, with a large concentration at four years and smaller groups at other commonly reported design lifetimes. Missing target values are therefore not imputed.

The full curated dataset is retained for possible unsupervised analysis, while only records with a reported expected lifetime are used for the supervised-learning task.

In [20]:
labeled_satellites_df = curated_satellites_df.loc[
    curated_satellites_df[TARGET].notna()
].copy()

unlabeled_satellites_df = curated_satellites_df.loc[
    curated_satellites_df[TARGET].isna()
].copy()

assert len(labeled_satellites_df) == 5351

print(f"Curated records:       {len(curated_satellites_df):,}")
print(f"Labeled records:       {len(labeled_satellites_df):,}")
print(f"Unlabeled records:     {len(unlabeled_satellites_df):,}")

Curated records:       7,270
Labeled records:       5,351
Unlabeled records:     1,919


## Candidate Feature Audit

The merged dataset contains many administrative, descriptive, and identifier fields that are not appropriate model inputs.

We therefore inspect a compact set of candidate variables describing orbital characteristics, physical properties, mission purpose, operator information, and launch timing. Identifiers, names, post-launch outcomes, and the target itself are excluded from consideration.

In [21]:
candidate_features = [
    "PERIOD",
    "INCLINATION",
    "APOGEE",
    "PERIGEE",
    "RCS_SIZE",
    "Launch Mass (kg.)",
    "Purpose",
    "Users",
    "Class of Orbit",
    "Type of Orbit",
    "Country of Operator/Owner",
    "LAUNCH_YEAR",
]

candidate_feature_audit = pd.DataFrame({
    "dtype": labeled_satellites_df[candidate_features].dtypes.astype(str),
    "missing": labeled_satellites_df[candidate_features].isna().sum(),
    "missing_pct": (
        labeled_satellites_df[candidate_features].isna().mean() * 100
    ).round(1),
    "unique": labeled_satellites_df[candidate_features].nunique(dropna=True),
})

display(candidate_feature_audit)

,dtype,missing,missing_pct,unique
PERIOD,float64,30,0.6,803
INCLINATION,float64,30,0.6,573
APOGEE,float64,30,0.6,765
PERIGEE,float64,30,0.6,764
RCS_SIZE,str,49,0.9,3
Launch Mass (kg.),float64,26,0.5,476
Purpose,str,0,0.0,17
Users,str,0,0.0,15
Class of Orbit,str,0,0.0,4
Type of Orbit,str,528,9.9,8


## Feature Selection

We retain a compact set of physical, orbital, mission, and launch-related variables for the predictive analysis.

Administrative identifiers and descriptive fields are excluded because they do not represent meaningful predictive characteristics. Variables that are redundant with retained features are also removed.

`CURRENT`, `DECAY`, and derived time-in-orbit variables are intentionally excluded because they depend on information observed after launch. Including them would provide the model with information that would not be available when predicting a satellite's expected lifetime.

`USERS`, `Type of Orbit`, and `Country of Operator/Owner` are not retained because the selected variables already provide the corresponding mission, orbital, and responsibility information in a more compact representation.

In [22]:
modeling_columns = [
    # Orbital characteristics
    "PERIOD",
    "INCLINATION",
    "APOGEE",
    "PERIGEE",

    # Physical characteristics
    "RCS_SIZE",
    "Launch Mass (kg.)",

    # Mission and orbit information
    "Purpose",
    "Class of Orbit",

    # Launch information
    "COUNTRY",
    "LAUNCH_YEAR",

    # Target
    "Expected Lifetime (yrs.)",
]

modeling_df = labeled_satellites_df[modeling_columns].copy()

modeling_df = modeling_df.rename(
    columns={
        "Launch Mass (kg.)": "LAUNCH_MASS_KG",
        "Purpose": "PURPOSE",
        "Class of Orbit": "CLASS_OF_ORBIT",
        "Expected Lifetime (yrs.)": "EXPECTED_LIFETIME_YRS",
    }
)

print(f"Modeling dataset shape: {modeling_df.shape}")
display(modeling_df.head())

Modeling dataset shape: (5351, 11)


,PERIOD,INCLINATION,APOGEE,PERIGEE,RCS_SIZE,LAUNCH_MASS_KG,PURPOSE,CLASS_OF_ORBIT,COUNTRY,LAUNCH_YEAR,EXPECTED_LIFETIME_YRS
0,NaN,NaN,NaN,NaN,NaN,14500.0,Earth Observation,LEO,US,2000,9.0
1,NaN,NaN,NaN,NaN,NaN,300.0,Earth Observation,LEO,ISRA,2007,5.0
2,NaN,NaN,NaN,NaN,NaN,770.0,Earth Observation,LEO,GER,2007,10.0
3,NaN,NaN,NaN,NaN,NaN,770.0,Earth Observation,LEO,GER,2008,10.0
4,NaN,NaN,NaN,NaN,NaN,770.0,Earth Observation,LEO,GER,2008,10.0


In [23]:
modeling_df["CLASS_OF_ORBIT"] = (
    modeling_df["CLASS_OF_ORBIT"]
    .str.strip()
    .str.upper()
)

modeling_df["PURPOSE"] = modeling_df["PURPOSE"].str.strip()
modeling_df["COUNTRY"] = modeling_df["COUNTRY"].str.strip()

print("Orbit classes:")
display(modeling_df["CLASS_OF_ORBIT"].value_counts(dropna=False))

print("\nCountries:")
display(modeling_df["COUNTRY"].value_counts().head(10))

Orbit classes:


CLASS_OF_ORBIT
LEO           4707
GEO            492
MEO            119
ELLIPTICAL      33
Name: count, dtype: int64


Countries:


COUNTRY
US      3893
UK       566
PRC      142
CIS      131
JPN       56
ESA       54
IND       50
SES       46
ORB       35
ITSO      34
Name: count, dtype: int64

### Grouping High-Cardinality Categorical Variables

`COUNTRY` contains many categories, while most records are concentrated in a small number of responsible country/entity codes.

The four dominant codes identified during exploratory analysis (`US`, `UK`, `PRC`, and `CIS`) are retained as individual categories, while the remaining values are grouped under `REST`.

In [24]:
top_countries = ["US", "UK", "PRC", "CIS"]

modeling_df["COUNTRY_GROUPED"] = (
    modeling_df["COUNTRY"]
    .where(modeling_df["COUNTRY"].isin(top_countries), "REST")
)

display(
    modeling_df["COUNTRY_GROUPED"]
    .value_counts()
    .rename("Satellites")
    .to_frame()
)

modeling_df = modeling_df.drop(columns="COUNTRY")

,Satellites
COUNTRY_GROUPED,
US,3893
REST,619
UK,566
PRC,142
CIS,131


`PURPOSE` contains several closely related and combined categories. To reduce sparsity before encoding, we consolidate them into a small number of broader mission groups.

The grouping preserves the main distinctions between communications, Earth observation, navigation, and scientific or technology-oriented missions. Rare categories that do not fit cleanly into one group are assigned to `OTHER`.

In [25]:
purpose_mapping = {
    "COMMUNICATIONS": [
        "Communications",
        "Communications/Technology Development",
    ],
    "EARTH_OBSERVATION": [
        "Earth Observation",
        "Earth Science",
        "Earth Observation/Technology Development",
        "Earth Observation/Space Science",
        "Earth Observation/Communications/Space Science",
        "Space Observation",
    ],
    "NAVIGATION": [
        "Navigation/Global Positioning",
        "Navigation/Regional Positioning",
        "Earth Observation/Navigation",
        "Communications/Navigation",
    ],
    "SCIENCE_TECHNOLOGY": [
        "Space Science",
        "Technology Development",
        "Technology Demonstration",
        "Meteorological",
    ],
}

inverse_purpose_mapping = {
    original: grouped
    for grouped, values in purpose_mapping.items()
    for original in values
}

modeling_df["PURPOSE_GROUPED"] = (
    modeling_df["PURPOSE"]
    .map(inverse_purpose_mapping)
    .fillna("OTHER")
)

display(
    modeling_df["PURPOSE_GROUPED"]
    .value_counts()
    .rename("Satellites")
    .to_frame()
)

modeling_df = modeling_df.drop(columns="PURPOSE")

,Satellites
PURPOSE_GROUPED,
COMMUNICATIONS,4601
EARTH_OBSERVATION,499
NAVIGATION,161
SCIENCE_TECHNOLOGY,90


## Missing Values

The selected modeling variables contain only a small number of missing values.

Before applying any imputation, we quantify the missing-data pattern. Imputation parameters are learned from the training set only, preventing information from the validation or test sets from influencing preprocessing.

In [26]:
missing_summary = pd.DataFrame({
    "missing": modeling_df.isna().sum(),
    "missing_pct": (modeling_df.isna().mean() * 100).round(2),
})

missing_summary = missing_summary.loc[
    missing_summary["missing"] > 0
]

display(missing_summary)

,missing,missing_pct
PERIOD,30,0.56
INCLINATION,30,0.56
APOGEE,30,0.56
PERIGEE,30,0.56
RCS_SIZE,49,0.92
LAUNCH_MASS_KG,26,0.49


In [27]:
orbital_columns = [
    "PERIOD",
    "INCLINATION",
    "APOGEE",
    "PERIGEE",
]

orbital_missing_pattern = (
    modeling_df[orbital_columns]
    .isna()
    .value_counts()
    .rename("Records")
    .to_frame()
)

display(orbital_missing_pattern)

,,,,Records
PERIOD,INCLINATION,APOGEE,PERIGEE,
False,False,False,False,5321
True,True,True,True,30


The four orbital variables are missing simultaneously in the same 30 records. Their missing fraction is small, as is the missingness in launch mass and radar cross-section size, so these values can be handled with simple robust imputations after the train/validation/test split.

## Train, Validation, and Test Split

The labeled dataset is divided into three subsets:

- 60% training data;
- 20% validation data;
- 20% test data.

The split is performed before any data-dependent preprocessing. This ensures that imputation, encoding, and scaling parameters are estimated using the training data only.

In [28]:
train_full_df, test_df = train_test_split(
    modeling_df,
    train_size=0.80,
    random_state=RANDOM_SEED,
)

train_df, validation_df = train_test_split(
    train_full_df,
    train_size=0.75,
    random_state=RANDOM_SEED,
)

print(f"Training records:   {len(train_df):,}")
print(f"Validation records: {len(validation_df):,}")
print(f"Test records:       {len(test_df):,}")

assert (
    len(train_df)
    + len(validation_df)
    + len(test_df)
    == len(modeling_df)
)

Training records:   3,210
Validation records: 1,070
Test records:       1,071


## Target Classes

The supervised-learning task is formulated as a three-class classification problem based on the reported expected lifetime of each satellite.

The class boundaries are defined using the mean and standard deviation of `EXPECTED_LIFETIME_YRS` in the training set only:

- **Short lifetime:** below $\mu - 0.5\sigma$
- **Medium lifetime:** between $\mu - 0.5\sigma$ and $\mu + 0.5\sigma$
- **Long lifetime:** above $\mu + 0.5\sigma$

These are dataset-derived classes rather than universal definitions of short, medium, and long satellite lifetime. Using the training set to determine the thresholds prevents validation and test information from influencing the target definition.

In [29]:
TARGET = "EXPECTED_LIFETIME_YRS"

target_mean = train_df[TARGET].mean()
target_std = train_df[TARGET].std()

low_threshold = target_mean - 0.5 * target_std
high_threshold = target_mean + 0.5 * target_std

print(f"Training mean:        {target_mean:.2f} years")
print(f"Training std:         {target_std:.2f} years")
print(f"Short/medium limit:   {low_threshold:.2f} years")
print(f"Medium/long limit:    {high_threshold:.2f} years")

Training mean:        5.37 years
Training std:         3.44 years
Short/medium limit:   3.65 years
Medium/long limit:    7.09 years


In [30]:
def assign_lifetime_class(value):
    if value < low_threshold:
        return "SHORT"
    if value > high_threshold:
        return "LONG"
    return "MEDIUM"


for df in [train_df, validation_df, test_df]:
    df["LIFETIME_CLASS"] = df[TARGET].apply(assign_lifetime_class)

In [31]:
class_distribution = pd.DataFrame({
    "Train": train_df["LIFETIME_CLASS"].value_counts(),
    "Validation": validation_df["LIFETIME_CLASS"].value_counts(),
    "Test": test_df["LIFETIME_CLASS"].value_counts(),
}).fillna(0).astype(int)

class_distribution = class_distribution.reindex(
    ["SHORT", "MEDIUM", "LONG"]
)

display(class_distribution)

display(
    class_distribution
    .div(class_distribution.sum(axis=0), axis=1)
    .mul(100)
    .round(1)
)

,Train,Validation,Test
LIFETIME_CLASS,,,
SHORT,252,84,89
MEDIUM,2514,830,840
LONG,444,156,142


,Train,Validation,Test
LIFETIME_CLASS,,,
SHORT,7.9,7.9,8.3
MEDIUM,78.3,77.6,78.4
LONG,13.8,14.6,13.3


The resulting class proportions are similar across the three subsets. The medium-lifetime class is the majority class, while short- and long-lifetime satellites are less frequent. Class imbalance is handled later at the modeling stage rather than by altering the curated data.

## Feature Preprocessing

The selected features require a small number of transformations before they can be used by the machine-learning models.

The preprocessing strategy is defined separately for each variable type:

- **Orbital variables** (`PERIOD`, `INCLINATION`, `APOGEE`, `PERIGEE`): only 30 records are missing these values, and the four variables are missing simultaneously. Since the missing fraction is very small (~0.6%), each variable is imputed using the median estimated from the training set. The median is preferred over the mean because orbital parameters can be strongly skewed by high-altitude or eccentric orbits.

- **Launch mass** (`LAUNCH_MASS_KG`): only 26 values are missing (~0.5%). These are also imputed with the training-set median. Given the small amount of missing data, a simple robust imputation is preferable to a more complex model-based method.

- **Radar cross-section size** (`RCS_SIZE`): 49 values are missing (~0.9%). Because this is a categorical variable with three naturally ordered levels, missing values are replaced with the most frequent training category. The categories are then ordinally encoded as `SMALL < MEDIUM < LARGE` and rescaled to the same 0–1 range used for numerical variables.

- **Nominal categorical variables** (`COUNTRY_GROUPED`, `PURPOSE_GROUPED`, `CLASS_OF_ORBIT`): these variables do not have a meaningful numerical ordering, so they are represented using one-hot encoding. Unknown categories encountered in validation or test data are ignored safely.

- **Numerical scaling**: numerical variables are rescaled with `MinMaxScaler`. The scaler is fitted only on the training set and then applied unchanged to validation and test data.

All imputation, encoding, and scaling parameters are learned from the training set only. This prevents information from the validation or test sets from leaking into preprocessing.

To apply these transformations consistently, the preprocessing steps are implemented with scikit-learn pipelines and a `ColumnTransformer`. This keeps transformations grouped by variable type and ensures that the same fitted preprocessing is applied to the validation and test sets.

In [32]:
feature_columns = [
    "PERIOD",
    "INCLINATION",
    "APOGEE",
    "PERIGEE",
    "LAUNCH_MASS_KG",
    "LAUNCH_YEAR",
    "RCS_SIZE",
    "COUNTRY_GROUPED",
    "PURPOSE_GROUPED",
    "CLASS_OF_ORBIT",
]

X_train = train_df[feature_columns].copy()
X_validation = validation_df[feature_columns].copy()
X_test = test_df[feature_columns].copy()

y_train = train_df["LIFETIME_CLASS"].copy()
y_validation = validation_df["LIFETIME_CLASS"].copy()
y_test = test_df["LIFETIME_CLASS"].copy()

In [33]:
numeric_features = [
    "PERIOD",
    "INCLINATION",
    "APOGEE",
    "PERIGEE",
    "LAUNCH_MASS_KG",
    "LAUNCH_YEAR",
]

ordinal_features = [
    "RCS_SIZE",
]

nominal_features = [
    "COUNTRY_GROUPED",
    "PURPOSE_GROUPED",
    "CLASS_OF_ORBIT",
]

In [34]:
numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", MinMaxScaler()),
    ]
)

rcs_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        (
            "encoder",
            OrdinalEncoder(
                categories=[["SMALL", "MEDIUM", "LARGE"]]
            ),
        ),
        ("scaler", MinMaxScaler()),
    ]
)

categorical_transformer = Pipeline(
    steps=[
        (
            "encoder",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False,
            ),
        ),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", numeric_transformer, numeric_features),
        ("rcs", rcs_transformer, ordinal_features),
        ("categorical", categorical_transformer, nominal_features),
    ],
    verbose_feature_names_out=False,
)

In [35]:
X_train_processed = preprocessor.fit_transform(X_train)

X_validation_processed = preprocessor.transform(X_validation)
X_test_processed = preprocessor.transform(X_test)

processed_feature_names = preprocessor.get_feature_names_out()

X_train_processed = pd.DataFrame(
    X_train_processed,
    columns=processed_feature_names,
    index=X_train.index,
)

X_validation_processed = pd.DataFrame(
    X_validation_processed,
    columns=processed_feature_names,
    index=X_validation.index,
)

X_test_processed = pd.DataFrame(
    X_test_processed,
    columns=processed_feature_names,
    index=X_test.index,
)

In [36]:
print(f"Training shape:   {X_train_processed.shape}")
print(f"Validation shape: {X_validation_processed.shape}")
print(f"Test shape:       {X_test_processed.shape}")

print(
    "\nMissing values after preprocessing:",
    X_train_processed.isna().sum().sum(),
    X_validation_processed.isna().sum().sum(),
    X_test_processed.isna().sum().sum(),
)

display(X_train_processed.head())

assert X_train_processed.shape[1] == X_validation_processed.shape[1]
assert X_train_processed.shape[1] == X_test_processed.shape[1]

assert X_train_processed.isna().sum().sum() == 0
assert X_validation_processed.isna().sum().sum() == 0
assert X_test_processed.isna().sum().sum() == 0

Training shape:   (3210, 20)
Validation shape: (1070, 20)
Test shape:       (1071, 20)

Missing values after preprocessing: 0 0 0


,PERIOD,INCLINATION,APOGEE,PERIGEE,LAUNCH_MASS_KG,LAUNCH_YEAR,RCS_SIZE,COUNTRY_GROUPED_CIS,COUNTRY_GROUPED_PRC,COUNTRY_GROUPED_REST,COUNTRY_GROUPED_UK,COUNTRY_GROUPED_US,PURPOSE_GROUPED_COMMUNICATIONS,PURPOSE_GROUPED_EARTH_OBSERVATION,PURPOSE_GROUPED_NAVIGATION,PURPOSE_GROUPED_SCIENCE_TECHNOLOGY,CLASS_OF_ORBIT_ELLIPTICAL,CLASS_OF_ORBIT_GEO,CLASS_OF_ORBIT_LEO,CLASS_OF_ORBIT_MEO
4114,0.000884,0.347389,0.001763,0.010549,0.012951,0.970588,1.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
3007,0.000896,0.346279,0.001786,0.010686,0.012951,0.941176,1.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
1897,0.001319,0.641775,0.002660,0.015873,0.006950,0.705882,0.5,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0
1362,0.000289,0.346084,0.000538,0.002877,0.012951,0.941176,1.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
4506,0.001951,0.573825,0.003921,0.023506,0.007350,1.000000,0.5,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0


After preprocessing, the training, validation, and test sets contain the same 20 model-ready features and no missing values.

The transformation from 10 input features to 20 model-ready columns is mainly produced by one-hot encoding the nominal categorical variables. The preprocessing pipeline is fitted exclusively on the training set and then applied unchanged to validation and test data.

## Final Modeling Datasets

The processed features are combined with their corresponding lifetime classes to create the training, validation, and test datasets used in the supervised-learning notebook.

Keeping these splits fixed ensures that all models are trained and evaluated on exactly the same observations.

In [37]:
train_model_df = X_train_processed.copy()
validation_model_df = X_validation_processed.copy()
test_model_df = X_test_processed.copy()

train_model_df["LIFETIME_CLASS"] = y_train.values
validation_model_df["LIFETIME_CLASS"] = y_validation.values
test_model_df["LIFETIME_CLASS"] = y_test.values

print("Training dataset:", train_model_df.shape)
print("Validation dataset:", validation_model_df.shape)
print("Test dataset:", test_model_df.shape)

display(train_model_df.head())

Training dataset: (3210, 21)
Validation dataset: (1070, 21)
Test dataset: (1071, 21)


,PERIOD,INCLINATION,APOGEE,PERIGEE,LAUNCH_MASS_KG,LAUNCH_YEAR,RCS_SIZE,COUNTRY_GROUPED_CIS,COUNTRY_GROUPED_PRC,COUNTRY_GROUPED_REST,COUNTRY_GROUPED_UK,COUNTRY_GROUPED_US,PURPOSE_GROUPED_COMMUNICATIONS,PURPOSE_GROUPED_EARTH_OBSERVATION,PURPOSE_GROUPED_NAVIGATION,PURPOSE_GROUPED_SCIENCE_TECHNOLOGY,CLASS_OF_ORBIT_ELLIPTICAL,CLASS_OF_ORBIT_GEO,CLASS_OF_ORBIT_LEO,CLASS_OF_ORBIT_MEO,LIFETIME_CLASS
4114,0.000884,0.347389,0.001763,0.010549,0.012951,0.970588,1.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,MEDIUM
3007,0.000896,0.346279,0.001786,0.010686,0.012951,0.941176,1.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,MEDIUM
1897,0.001319,0.641775,0.002660,0.015873,0.006950,0.705882,0.5,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,SHORT
1362,0.000289,0.346084,0.000538,0.002877,0.012951,0.941176,1.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,MEDIUM
4506,0.001951,0.573825,0.003921,0.023506,0.007350,1.000000,0.5,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,MEDIUM


In [38]:
PROCESSED_DATA_DIR = Path("../data/processed")
PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)

train_model_df.to_csv(
    PROCESSED_DATA_DIR / "train.csv",
    index=False,
)

validation_model_df.to_csv(
    PROCESSED_DATA_DIR / "validation.csv",
    index=False,
)

test_model_df.to_csv(
    PROCESSED_DATA_DIR / "test.csv",
    index=False,
)

for filename in ["train.csv", "validation.csv", "test.csv"]:
    path = PROCESSED_DATA_DIR / filename
    print(f"{filename}: {path.stat().st_size / 1024:.1f} KB")

train.csv: 573.4 KB
validation.csv: 191.3 KB
test.csv: 191.7 KB


## Summary

This curation workflow produces a validated and reproducible dataset for satellite lifetime classification.

The main steps were:

- restrict the population to payloads;
- clean duplicate UCS records and validate cross-dataset links using NORAD and COSPAR identifiers;
- preserve only satellites with reported expected lifetime for supervised learning;
- select a compact set of physical, orbital, mission, and launch-related features;
- group high-cardinality categorical variables into interpretable categories;
- split the labeled data before fitting any preprocessing transformation;
- define short-, medium-, and long-lifetime classes using training-set statistics only;
- impute, encode, and scale features using transformations fitted exclusively on the training set;
- export fixed training, validation, and test datasets for downstream modeling.